In [1]:
#Environment Setup & Repository Cloning
import os

print("Cloning Harmonia-LM...")
!git clone https://github.com/Katsukuten/Harmonia-LM.git
%cd Harmonia-LM

print("Installing dependencies...")
!pip install -q miditok transformers pytorch-lightning seaborn matplotlib
print("Environment ready.")

Cloning Harmonia-LM...
Cloning into 'Harmonia-LM'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 267 (delta 66), reused 9 (delta 9), pack-reused 164 (from 1)
Receiving objects: 100% (267/267), 6.25 MiB | 13.17 MiB/s, done.
Resolving deltas: 100% (154/154), done.
/content/Harmonia-LM
Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.2/852.2 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.9 MB/s eta 0:00:00
Environment ready.


In [3]:
# Download Weights & Load Model
import torch
import os
import sys
from pathlib import Path
sys.path.append(os.path.abspath('model'))
from miditok import TSD
from training import Qwen3MusicModel

# --- Public Direct Links ---
CKPT_URL = "https://github.com/Katsukuten/Harmonia-LM/releases/download/v1.0/last_4096.ckpt"
TOKENIZER_URL = "https://github.com/Katsukuten/Harmonia-LM/releases/download/v1.0/tokenizer.json"

print("Downloading model weights and tokenizer directly from GitHub...")
os.system(f"wget -q -O best_model.ckpt {CKPT_URL}")
os.system(f"wget -q -O tokenizer.json {TOKENIZER_URL}")

print("Loading Tokenizer...")
tokenizer = TSD(params=Path("tokenizer.json"))

print("Loading model weights in VRAM...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PyTorch 2.6 security bypass
_original_load = torch.load
def _trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _trusted_load

try:
    model = Qwen3MusicModel.load_from_checkpoint(
        Path("best_model.ckpt"),
        tokenizer=tokenizer,
        tokenizer_vocab_size=tokenizer.vocab_size
    )
finally:
    torch.load = _original_load

model.eval().to(device)
print(f"Model successfully loaded on: {device}")

Loading Tokenizer...
Loading model weights in VRAM...
Model successfully loaded on: cuda


In [4]:
# @title 4. Generation Engine (With Interactive Primer Upload)
from transformers import GenerationConfig
from copy import deepcopy
from google.colab import files
import os

# @markdown ### ⚙️ Generation Parameters
Use_Primer = True # @param {type:"boolean"}
Primer_Length = 512 # @param {type:"integer"}
Generation_Length = 1024 # @param {type:"integer"}
TEMPERATURE = 0.95 # @param {type:"slider", min:0.1, max:1.5, step:0.05}
TOP_P = 0.95 # @param {type:"slider", min:0.5, max:1.0, step:0.01}
REPETITION_PENALTY = 1.05

primer_midi_path = None
primer_len = 0

# Primer Upload Logic
if Use_Primer:
    print("Please upload a .mid or .midi file from your computer:")
    uploaded = files.upload()
    if uploaded:
        primer_midi_path = list(uploaded.keys())[0]
        primer_len = Primer_Length
        print(f"✅ Primer loaded: {primer_midi_path}")
    else:
        print("⚠️ No file uploaded. Defaulting to Ex Nihilo (From scratch).")

# Tokenization & Input Preparation
bos_token_id = tokenizer["BOS_None"] if "BOS_None" in tokenizer.vocab else tokenizer["PAD_None"]

if primer_midi_path:
    primer_tokens = tokenizer(primer_midi_path)
    ids = primer_tokens[0].ids if hasattr(primer_tokens[0], 'ids') else primer_tokens[0]

    if len(ids) > primer_len:
        ids = ids[:primer_len]

    if ids[0] != bos_token_id:
        ids.insert(0, bos_token_id)

    input_ids = torch.tensor([ids], dtype=torch.long).to(device)
    print(f"Primer context set to {len(ids)} tokens.")
else:
    input_ids = torch.tensor([[bos_token_id]], dtype=torch.long).to(device)
    print("Generating from blank state (BOS).")

# Generation Config
gen_config = GenerationConfig(
    max_new_tokens=Generation_Length,
    do_sample=True,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=0,
    eos_token_id=tokenizer["EOS_None"] if "EOS_None" in tokenizer.vocab else tokenizer["PAD_None"],
    pad_token_id=tokenizer["PAD_None"],
    repetition_penalty=REPETITION_PENALTY
)

# Inference
print(f"Generating sequence... (Temp: {TEMPERATURE}, Top-P: {TOP_P})")
with torch.no_grad():
    outputs = model.model.generate(
        inputs=input_ids,
        generation_config=gen_config,
        use_cache=True,
    )

generated_ids = outputs[0].tolist()
print(f"Generation complete! Total length: {len(generated_ids)} tokens.")

# Decode & Download
midi_obj = tokenizer.decode([deepcopy(generated_ids)])
midi_bytes = midi_obj.dumps_midi()
midi_path = "output_harmonia.mid"

with open(midi_path, "wb") as f:
    f.write(midi_bytes)

print("Downloading your generated MIDI track...")
files.download(midi_path)

Please upload a .mid or .midi file from your computer:


Saving debussy-clair-de-lune.mid to debussy-clair-de-lune.mid
✅ Primer loaded: debussy-clair-de-lune.mid
Primer context set to 513 tokens.
Generating sequence... (Temp: 0.95, Top-P: 0.95)
Generation complete! Total length: 1537 tokens.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>